# Model Optimization - Bot Detection
## Hyperparameter Tuning & Advanced Optimization

## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold,
                                     learning_curve, validation_curve)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, make_scorer)
import joblib
import os
from scipy.stats import randint, uniform

# Configuration
#plt.style.use('seaborn')
sns.set_palette('pastel')
%matplotlib inline

# Create output directories
os.makedirs('../models/optimized', exist_ok=True)
os.makedirs('../reports/optimization_plots', exist_ok=True)

## 2. Data Loading

In [1]:
# Load processed data
X_train = pd.read_csv('../data/processed/X_train.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

## 3. Optimization Strategy

In [2]:
# Define common parameters
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {'AUC': 'roc_auc', 'F1': make_scorer(roc_auc_score)}

## 4. Random Forest Optimization

In [3]:
# Define parameter grid
rf_params = {
    'n_estimators': randint(100, 500),
    'max_depth': [None] + list(np.arange(5, 50, 5)),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None],
    'class_weight': ['balanced', {0:1, 1:3}]
}

# Initialize search
rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=rf_params,
    n_iter=100,
    scoring=scoring,
    refit='F1',
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Execute search
rf_search.fit(X_train, y_train)

# Save best model
joblib.dump(rf_search.best_estimator_, '../models/optimized/rf_optimized.pkl')

Fitting 5 folds for each of 100 candidates, totalling 500 fits


['../models/optimized/rf_optimized.pkl']

## 5. XGBoost Optimization

In [4]:
# Define parameter grid
xgb_params = {
    'learning_rate': uniform(0.01, 0.3),
    'max_depth': randint(3, 15),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 1),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1),
    'scale_pos_weight': [3.0, 3.4, 4.0]
}

# Initialize search
xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    param_distributions=xgb_params,
    n_iter=100,
    scoring=scoring,
    refit='F1',
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Execute search
xgb_search.fit(X_train, y_train)

# Save best model
joblib.dump(xgb_search.best_estimator_, '../models/optimized/xgb_optimized.pkl')

Fitting 5 folds for each of 100 candidates, totalling 500 fits


['../models/optimized/xgb_optimized.pkl']

## 6. SVM Optimization

In [ ]:
# Define parameter grid
svm_params = {
    'C': uniform(0.1, 10),
    'kernel': ['linear', 'poly', 'rbf'],
    'degree': randint(2, 5),
    'gamma': ['scale', 'auto'] + list(np.logspace(-3, 3, 7))
}

# Initialize search
svm_search = RandomizedSearchCV(
    estimator=SVC(class_weight='balanced', probability=True, random_state=42),
    param_distributions=svm_params,
    n_iter=50,
    scoring=scoring,
    refit='F1',
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Execute search
svm_search.fit(X_train, y_train)

# Save best model
joblib.dump(svm_search.best_estimator_, '../models/optimized/svm_optimized.pkl')

Fitting 5 folds for each of 50 candidates, totalling 250 fits


## 7. Optimization Analysis

In [ ]:
def plot_optimization_results(search, model_name):
    # Learning Curve
    train_sizes, train_scores, test_scores = learning_curve(
        search.best_estimator_, X_train, y_train, cv=cv,
        scoring='roc_auc', n_jobs=-1
    )
    
    plt.figure(figsize=(12, 6))
    plt.plot(train_sizes, np.mean(train_scores, axis=1), label='Training Score')
    plt.plot(train_sizes, np.mean(test_scores, axis=1), label='CV Score')
    plt.title(f'{model_name} Learning Curve')
    plt.xlabel('Training Examples')
    plt.ylabel('AUC-ROC Score')
    plt.legend()
    plt.savefig(f'../reports/optimization_plots/{model_name}_learning_curve.png')
    plt.show()

# Plot for each model
plot_optimization_results(rf_search, 'RandomForest')
plot_optimization_results(xgb_search, 'XGBoost')
plot_optimization_results(svm_search, 'SVM')

## 8. Final Model Evaluation

In [ ]:
optimized_models = {
    'Random Forest': joblib.load('../models/optimized/rf_optimized.pkl'),
    'XGBoost': joblib.load('../models/optimized/xgb_optimized.pkl'),
    'SVM': joblib.load('../models/optimized/svm_optimized.pkl')
}

results = []

for name, model in optimized_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    })

# Compare with baseline
baseline_results = pd.read_csv('../reports/baseline_metrics.csv')
optimized_results = pd.DataFrame(results).set_index('Model')

print("\n=== Performance Comparison ===")
comparison = pd.concat([baseline_results, optimized_results], 
                      keys=['Baseline', 'Optimized'], axis=1)
display(comparison.style.format("{:.3f}").background_gradient(cmap='Blues'))

## 9. Best Model Deployment

In [ ]:
# Identify best model
best_model = optimized_results.idxmax()['F1']
print(f"\nBest Model: {best_model}")

# Save final model
joblib.dump(optimized_models[best_model], '../models/final_model.pkl')

## 10. Next Steps

**Recommended Actions**:
1. Conduct feature engineering on top predictors
2. Implement ensemble methods (stacking/voting)
3. Perform threshold optimization using precision-recall curves
4. Set up monitoring for concept drift